# DFT 상대에너지와 실제 금속 결정구조 분석

이 노트북은 `output/delta_e_transition_dataset.csv`를 이용해 다음 질문을 정량적으로 분석합니다.

1. DFT/OQMD 상대에너지는 실제 상온 결정구조를 얼마나 잘 맞히는가?
2. 예측이 틀린 원소들은 어떤 특징을 가지는가?
3. 구조 간 에너지 차이 `Delta_E`가 작은 원소일수록 상전이와 관련이 있는가?
4. 원소 물성 중 `Delta_E`와 관련이 큰 변수는 무엇인가?

> 주의: 현재 OQMD `formationenergy` API에서 얻은 값은 total electronic energy가 아니라 주로 `delta_e` 기반 상대에너지입니다. 따라서 해석은 “OQMD 상대에너지 기준의 구조 안정성 비교”로 표현하는 것이 안전합니다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.dpi"] = 130

DATA_PATH = Path("output/delta_e_transition_dataset.csv")
FIG_DIR = Path("output/figures")
TABLE_DIR = Path("output/analysis_tables")
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.head()

## 1. 데이터 기본 확인

먼저 원소 수, 결측치, 주요 열을 확인합니다. `E_BCC`, `E_FCC`, `E_HCP`, `Delta_E`에 결측치가 있으면 구조 비교 분석에서 제외해야 합니다.

In [ ]:
print(f"원소 수: {len(df)}")
print("\n열 목록:")
print(df.columns.tolist())

key_cols = ["Element", "Actual Structure", "DFT Stable Structure", "E_BCC", "E_FCC", "E_HCP", "Delta_E", "Has Transition"]
df[key_cols].head(10)

In [ ]:
missing_summary = df[key_cols].isna().sum().to_frame("Missing Count")
missing_summary.to_csv(TABLE_DIR / "missing_summary.csv", encoding="utf-8-sig")
missing_summary

## 2. DFT 예측 구조와 실제 구조 비교

`DFT Matches Actual`의 평균은 정확도입니다. 원소 수가 많지 않으므로 정확도와 함께 불일치 원소를 표로 제시하는 것이 좋습니다.

In [ ]:
valid_pred = df.dropna(subset=["Actual Structure", "DFT Stable Structure"]).copy()
valid_pred["DFT Matches Actual"] = valid_pred["Actual Structure"] == valid_pred["DFT Stable Structure"]

accuracy = valid_pred["DFT Matches Actual"].mean()
n_correct = int(valid_pred["DFT Matches Actual"].sum())
n_total = len(valid_pred)

accuracy_table = pd.DataFrame({
    "Metric": ["Correct", "Total", "Accuracy"],
    "Value": [n_correct, n_total, accuracy],
})
accuracy_table.to_csv(TABLE_DIR / "prediction_accuracy.csv", index=False, encoding="utf-8-sig")
accuracy_table

In [ ]:
mismatch = valid_pred.loc[~valid_pred["DFT Matches Actual"], [
    "Element", "Actual Structure", "DFT Stable Structure", "Second Stable Structure",
    "E_BCC", "E_FCC", "E_HCP", "Delta_E", "Has Transition", "Transition Temp C"
]].sort_values("Delta_E")

mismatch.to_csv(TABLE_DIR / "mismatched_elements.csv", index=False, encoding="utf-8-sig")
mismatch

In [ ]:
labels = ["BCC", "FCC", "HCP"]
cm = confusion_matrix(valid_pred["Actual Structure"], valid_pred["DFT Stable Structure"], labels=labels)
cm_df = pd.DataFrame(cm, index=[f"Actual {x}" for x in labels], columns=[f"DFT {x}" for x in labels])
cm_df.to_csv(TABLE_DIR / "confusion_matrix.csv", encoding="utf-8-sig")

plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title(f"Actual vs DFT Stable Structure\nAccuracy = {accuracy:.1%}")
plt.tight_layout()
plt.savefig(FIG_DIR / "confusion_matrix.png", bbox_inches="tight")
plt.show()

cm_df

## 3. Delta_E 분포

`Delta_E = 두 번째 안정 구조 에너지 - 최안정 구조 에너지`입니다. 값이 작을수록 두 구조가 거의 비슷하게 안정하다는 뜻입니다.

In [ ]:
delta_summary = df["Delta_E"].describe().to_frame("Delta_E")
delta_summary.to_csv(TABLE_DIR / "delta_e_summary.csv", encoding="utf-8-sig")
delta_summary

In [ ]:
# Figure 5. Delta_E distribution histogram
# Large standalone version, same style as the original graph
plt.figure(figsize=(10, 6))
sns.histplot(df["Delta_E"].dropna(), bins=12, kde=True, color="#4C78A8", alpha=0.45, edgecolor="white")
plt.axvline(
    df["Delta_E"].median(),
    color="black",
    linestyle="--",
    linewidth=1.8,
    label=f"Median = {df['Delta_E'].median():.4f}"
)
plt.xlabel("Delta_E (eV/atom)", fontsize=14)
plt.ylabel("Count", fontsize=14)
plt.title("Distribution of Delta_E", fontsize=16)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "figure5_delta_e_distribution.png", bbox_inches="tight", dpi=400)
plt.show()


In [ ]:
small_delta = df.sort_values("Delta_E").loc[:, [
    "Element", "Actual Structure", "DFT Stable Structure", "Second Stable Structure",
    "Delta_E", "Has Transition", "Transition Temp C"
]].head(10)
small_delta.to_csv(TABLE_DIR / "smallest_delta_e_elements.csv", index=False, encoding="utf-8-sig")
small_delta

보고서 해석 포인트:

- `Delta_E`가 매우 작은 원소는 구조 간 안정성 차이가 작습니다.
- 이런 원소에서는 DFT 상대에너지, 온도, 엔트로피, 실험 조건에 따라 안정 구조 판단이 민감해질 수 있습니다.
- 불일치 원소가 작은 `Delta_E` 영역에 모이면, “구조 간 에너지 차이가 작을수록 DFT 예측과 실제 구조가 어긋나기 쉽다”는 해석이 가능합니다.

## 4. 상전이 여부와 Delta_E 관계

가설: `Delta_E`가 작은 원소일수록 구조 간 에너지 장벽 또는 안정성 차이가 작아 상전이가 일어나기 쉬울 수 있다.

단, Fe처럼 자기 엔트로피와 진동 엔트로피의 영향이 큰 원소는 `Delta_E`만으로 설명되지 않을 수 있습니다.

In [ ]:
transition_stats = df.groupby("Has Transition", dropna=False)["Delta_E"].agg([
    "count", "mean", "median", "std", "min", "max"
]).reset_index()
transition_stats.to_csv(TABLE_DIR / "delta_e_by_transition.csv", index=False, encoding="utf-8-sig")
transition_stats

In [ ]:
plt.figure(figsize=(6.5, 4.5))
sns.boxplot(data=df, x="Has Transition", y="Delta_E", hue="Has Transition", palette="Set2", legend=False)
sns.stripplot(data=df, x="Has Transition", y="Delta_E", color="black", alpha=0.65, jitter=0.12)
plt.xlabel("Has Phase Transition")
plt.ylabel("Delta_E (eV/atom)")
plt.title("Delta_E by Phase Transition Existence")
plt.tight_layout()
plt.savefig(FIG_DIR / "delta_e_by_transition_boxplot.png", bbox_inches="tight")
plt.show()

In [ ]:
# Figure 6. Delta_E comparison by element
# Large standalone version, keeping the original vertical-bar style
plot_df = df.sort_values("Delta_E").copy()

plt.figure(figsize=(13, 6.5))
sns.barplot(
    data=plot_df,
    x="Element",
    y="Delta_E",
    hue="Has Transition",
    dodge=False,
    palette="Set2"
)
plt.xticks(rotation=45, ha="right", fontsize=11)
plt.yticks(fontsize=12)
plt.xlabel("Element", fontsize=14)
plt.ylabel("Delta_E (eV/atom)", fontsize=14)
plt.title("Elements Ordered by Delta_E", fontsize=16)
plt.legend(title="Has Transition", fontsize=11, title_fontsize=12, loc="upper left")
plt.tight_layout()
plt.savefig(FIG_DIR / "figure6_elements_ordered_by_delta_e.png", bbox_inches="tight", dpi=400)
plt.show()


## 5. Fe 집중 분석

Fe는 연구 주제의 대표 사례입니다. DFT 상대에너지 기준으로는 BCC가 안정하지만, 실제로는 912°C에서 FCC로 상전이합니다. 이는 고온에서 Gibbs 자유에너지의 엔트로피 항이 중요해진다는 해석으로 연결됩니다.

In [ ]:
fe = df[df["Element"] == "Fe"].T
fe.to_csv(TABLE_DIR / "fe_case.csv", encoding="utf-8-sig")
fe

In [ ]:
# Fe case plot: large standalone version
fe = df[df["Element"] == "Fe"].iloc[0]
fe_long = pd.DataFrame({
    "Structure": ["BCC", "FCC", "HCP"],
    "Relative Energy": [fe["E_BCC"], fe["E_FCC"], fe["E_HCP"]],
})

plt.figure(figsize=(8, 5.5))
sns.barplot(data=fe_long, x="Structure", y="Relative Energy", palette="Set2", hue="Structure", legend=False)
plt.ylabel("Relative Energy (eV/atom)", fontsize=14)
plt.xlabel("Structure", fontsize=14)
plt.title("Fe: Relative Energy by Structure", fontsize=16)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "fe_structure_energy_large.png", bbox_inches="tight", dpi=400)
plt.show()

fe_long


Fe 해석 문장 예시:

> Fe는 OQMD 상대에너지 기준에서 BCC가 가장 안정하여 실제 상온 구조와 일치한다. 그러나 912°C에서 FCC로 상전이하므로, 고온 안정성은 0 K 전자에너지뿐 아니라 진동 엔트로피 및 자기 엔트로피가 포함된 Gibbs 자유에너지에 의해 결정됨을 보여준다.

## 6. 원소 물성과 Delta_E 관계

원자 반지름, 전기음성도, 이온화에너지, 밀도, 녹는점, d 전자 수가 `Delta_E`와 어떤 관계를 가지는지 봅니다.

In [ ]:
features = [
    "Atomic Radius pm",
    "Electronegativity",
    "First Ionization Energy kJ mol-1",
    "Density g cm-3",
    "Melting Point C",
    "d Electrons",
]

corr_cols = features + ["Delta_E"]
corr = df[corr_cols].corr(numeric_only=True)
corr.to_csv(TABLE_DIR / "property_delta_e_correlation.csv", encoding="utf-8-sig")

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Correlation Between Properties and Delta_E")
plt.tight_layout()
plt.savefig(FIG_DIR / "property_delta_e_correlation.png", bbox_inches="tight")
plt.show()

corr[["Delta_E"]].sort_values("Delta_E", ascending=False)

In [ ]:
scatter_features = ["d Electrons", "Atomic Radius pm", "Melting Point C", "Density g cm-3"]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.ravel()

for ax, feature in zip(axes, scatter_features):
    sns.scatterplot(data=df, x=feature, y="Delta_E", hue="Has Transition", style="DFT Matches Actual", s=80, ax=ax)
    for _, row in df.iterrows():
        if pd.notna(row[feature]) and pd.notna(row["Delta_E"]):
            ax.text(row[feature], row["Delta_E"], row["Element"], fontsize=8, alpha=0.75)
    ax.set_title(f"Delta_E vs {feature}")

plt.tight_layout()
plt.savefig(FIG_DIR / "delta_e_vs_properties.png", bbox_inches="tight")
plt.show()

## 7. Random Forest Feature Importance

데이터가 28개로 작기 때문에 예측 성능 자체보다, 어떤 물성이 `Delta_E` 설명에 상대적으로 많이 사용되는지 참고용으로 봅니다.

In [ ]:
model_df = df.dropna(subset=features + ["Delta_E"]).copy()
X = model_df[features]
y = model_df["Delta_E"]

if len(model_df) >= 10:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    rf = RandomForestRegressor(n_estimators=500, random_state=42)
    rf.fit(X_train, y_train)
    
    importance = pd.DataFrame({"Feature": features, "Importance": rf.feature_importances_})
    importance = importance.sort_values("Importance", ascending=False)
    importance.to_csv(TABLE_DIR / "random_forest_feature_importance.csv", index=False, encoding="utf-8-sig")
    
    print(f"Train R^2: {rf.score(X_train, y_train):.3f}")
    print(f"Test R^2 : {rf.score(X_test, y_test):.3f}")
    display(importance)
    
    plt.figure(figsize=(7, 4.5))
    sns.barplot(data=importance, x="Importance", y="Feature", color="#59A14F")
    plt.title("Random Forest Feature Importance for Delta_E")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "random_forest_feature_importance.png", bbox_inches="tight")
    plt.show()
else:
    print("데이터 수가 너무 적어 Random Forest 분석을 생략합니다.")

## 8. 선택: SHAP Summary Plot

`shap` 패키지가 설치되어 있으면 실행됩니다. 설치되어 있지 않으면 이 셀은 건너뜁니다.

```powershell
pip install shap
```

In [ ]:
try:
    import shap
    
    if "rf" in globals():
        explainer = shap.TreeExplainer(rf)
        shap_values = explainer.shap_values(X)
        shap.summary_plot(shap_values, X, show=False)
        plt.tight_layout()
        plt.savefig(FIG_DIR / "shap_summary.png", bbox_inches="tight")
        plt.show()
    else:
        print("Random Forest 모델이 없어 SHAP 분석을 생략합니다.")
except ImportError:
    print("shap 패키지가 설치되어 있지 않아 SHAP 분석을 생략합니다.")

## 9. 보고서용 핵심 표 생성

아래 표는 보고서 본문에 넣기 좋은 핵심 열만 모은 것입니다.

In [ ]:
report_table = df[[
    "Element", "Actual Structure", "DFT Stable Structure", "DFT Matches Actual",
    "E_BCC", "E_FCC", "E_HCP", "Delta_E",
    "Has Transition", "Transition Temp C", "Before", "After", "d Electrons"
]].sort_values(["DFT Matches Actual", "Delta_E"], ascending=[True, True])

report_table.to_csv(TABLE_DIR / "report_core_table.csv", index=False, encoding="utf-8-sig")
report_table

## 10. 결론 작성 가이드

분석 결과를 바탕으로 다음 구조로 결론을 쓰면 좋습니다.

1. DFT/OQMD 상대에너지는 전체 금속 중 상당수의 상온 결정구조를 맞혔다.
2. 예측 실패 원소는 대체로 `Delta_E`가 매우 작아 구조 간 에너지 차이가 거의 없었다.
3. 따라서 `Delta_E`가 작은 원소에서는 구조 안정성 판단이 계산 조건이나 온도 효과에 민감할 수 있다.
4. Fe는 상온 BCC 구조를 DFT가 맞히지만, 912°C에서 FCC로 상전이하므로 Electronic Energy만으로 고온 상안정성을 완전히 설명할 수 없다.
5. 실제 상안정성은 Gibbs 자유에너지로 결정되며, 진동 엔트로피와 자기 엔트로피가 중요한 역할을 할 수 있다.